# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` identifiers.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata overview
metadata = dataset.metadata
print('Dataset title:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))

## 2. Data Overview
Discover the available record sets, fields, and their `@id` values.

You may inspect the dataset's structure to find which record sets exist and how the data is organized.
Below, we list all available record sets along with their `@id`, and for each record set, their fields and corresponding `@id` and names.

In [ ]:
# Find all record sets in the Croissant metadata
record_sets = []
for obj in getattr(metadata, 'recordSet', []):
    rset = dataset.metadata.resolve(obj)
    record_sets.append(rset)

if not record_sets:
    # If no record sets in main metadata, search via the API (common for Croissant API datasets)
    from itertools import chain
    if hasattr(dataset, '_schema'):
        objs = dataset._schema.get('@graph', [])
    else:
        objs = []
    # Find all objects of type 'cr:RecordSet' or 'RecordSet'
    candidates = []
    for obj in objs:
        if obj.get('@type') in ['cr:RecordSet', 'RecordSet', 'http://mlcommons.org/croissant/RecordSet']:
            candidates.append(obj)
    record_sets = candidates

record_set_ids = []

print('Available record sets and fields:')
for rset in record_sets:
    rset_id = rset.get('@id') if isinstance(rset, dict) else getattr(rset, '@id', None)
    rset_name = rset.get('name', None) if isinstance(rset, dict) else getattr(rset, 'name', None)
    print(f' - RecordSet: {rset_name} (@id: {rset_id})')
    fields = rset.get('field', []) if isinstance(rset, dict) else getattr(rset, 'field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_obj = field
        else:
            # If field is an @id, resolve
            field_obj = dataset.metadata.resolve(field)
        field_id = field_obj.get('@id', None)
        field_name = field_obj.get('name', None)
        print(f'    - Field: {field_name} (@id: {field_id})')
    record_set_ids.append(rset_id)

## 3. Data Extraction
Now we load the data from one or more record sets into Pandas DataFrames for analysis.
Remember: the `record_set` parameter should always be referenced using the record set's `@id` (as returned above).

Below, we load the *first* available record set for demonstration, but you can extract more as needed.

In [ ]:
# Use the @id of the record set(s) you want to extract
selected_record_sets = record_set_ids[:1]  # Select first record set for example

dataframes = {}
for rset_id in selected_record_sets:
    print(f'Extracting records from record set @id: {rset_id}')
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
    else:
        print(f'No records found for record set {rset_id}')

# List the columns of the first DataFrame
if dataframes:
    first_rset_id = selected_record_sets[0]
    print(f"Columns in record set {first_rset_id}:")
    print(dataframes[first_rset_id].columns.tolist())
    display(dataframes[first_rset_id].head())

## 4. Exploratory Data Analysis (EDA)
Example EDA: filter records, normalize a numeric field and analyze group statistics.

Remember to use field or column `@id` for referencing!

In [ ]:
# List all columns to help pick suitable fields
if dataframes:
    df = dataframes[first_rset_id]
    print('Available DataFrame columns:')
    print(df.columns.tolist())
else:
    print('No data frame loaded to proceed with EDA.')

In [ ]:
# Choose a numeric field and a grouping field by their @id.
# Please adapt these @ids (column names) depending on what was shown above.
numeric_field_id = None
group_field_id = None

# Example guessing (replace with correct IDs after reviewing columns above):
for col in df.columns:
    if numeric_field_id is None and ('age' in col.lower() or 'interval' in col.lower()):
        numeric_field_id = col  # e.g., '@id: age' or similar
    if group_field_id is None and ('sex' in col.lower() or 'gender' in col.lower() or 'msi' in col.lower()):
        group_field_id = col

print(f'Using numeric field: {numeric_field_id}')
print(f'Using group field: {group_field_id}')

if numeric_field_id and numeric_field_id in df.columns:
    # Remove NaN and filter on numeric threshold
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.1f} (mean):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group stats
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
    else:
        print('Group field not found or not chosen.')
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize the distribution of the numeric field and its relationship with the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and preview the FAIR² dataset, identified relevant record sets and fields using their `@id`, and performed basic EDA and visualization. You can adapt this template to conduct more advanced analyses or apply it to similar datasets defined by a Croissant schema.